# EduGenie 🧞 — AI Learning Assistant (Google Colab Edition) — Full Quality version

Free, open-source Hugging Face models running **locally** — **no API keys required.**
This version uses the **full-size, higher-quality models** (better answers than the lite/fast version), which takes a bit longer to download the first time — that's expected and fine.

## ⚠️ How to run this notebook

Only **2 code cells**. Run them **in order**, and wait for each to finish before running the next:

1. **Cell 1 (Setup)** — click it, press **Shift+Enter**, wait for "Setup done."
2. **Cell 2 (everything else)** — click it, press **Shift+Enter**. First run downloads ~2 GB of models, so it can take **a few minutes** depending on your connection — this is normal, not stuck. Watch the printed progress lines (`Loading ...`, then `loaded: ...` for each model). When you see **"All models loaded successfully"**, the widget UI appears right below — that's when you can start using it.

**If a popup appears** saying "Warning: this notebook was not authored by Google" — click **Run anyway**. That's the #1 reason later cells look like they never run.

**Do not click the cell again or stop it while it's running** (the small red spinning circle next to the cell number means it's actively working, not stuck) — interrupting it just makes it start over.


## Cell 1 — Setup (run this first)

In [ ]:
# IMPORTANT: `pip install -U transformers` can pull an unstable pre-release
# (transformers 5.x), which REMOVED the text2text-generation / summarization /
# plain question-answering pipeline tasks entirely -> causes:
#   KeyError: "Unknown task text2text-generation, available tasks are [...]"
# Fix: pin to the stable 4.x line, which still supports all the tasks we use.
#
# We still do NOT touch torch/torchvision -- Colab's preinstalled pair already works.
!pip install -q "transformers<5" "huggingface_hub<1" sentencepiece accelerate ipywidgets

import torch
import transformers
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("Setup done. Now run Cell 2 below.")


## Cell 2 — Load models + define all modules + show the app

Models used: `google/flan-t5-base` (Explanation, generative Q&A, Quiz, Learning Path), `distilbert-base-cased-distilled-squad` (extractive Q&A over a context you give), and `facebook/bart-large-cnn` (Summarization) — all public, free, key-free models from the Hugging Face Hub.

In [ ]:
import torch
from transformers import pipeline
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import time


# ---------- 1. Load models ----------
device = 0 if torch.cuda.is_available() else -1
print("Using GPU" if device == 0 else "Using CPU")

def safe_load(task, model_name):
    t0 = time.time()
    print(f"Loading {model_name} ...")
    p = pipeline(task, model=model_name, device=device)
    print(f"  loaded: {model_name}  ({time.time()-t0:.1f}s)")
    return p

# General-purpose instruction-following model -> Explain, generative Q&A, Quiz, Learning Path
text2text = safe_load("text2text-generation", "google/flan-t5-base")

# Extractive Q&A model -> used when the user supplies a context passage
qa_pipeline = safe_load("question-answering", "distilbert-base-cased-distilled-squad")

# Summarization model
summarizer = safe_load("summarization", "facebook/bart-large-cnn")

print("All models loaded successfully\n")


# ---------- 2. Explanation Module (/explain) ----------
def explain_topic(topic: str, level: str = "beginner") -> str:
    prompt = (
        f"Explain the concept of '{topic}' to a {level} student. "
        f"Use simple language, a short analogy, and keep it under 150 words."
    )
    result = text2text(prompt, max_new_tokens=220, do_sample=False)
    return result[0]["generated_text"].strip()


# ---------- 3. Q&A Module (/qa) ----------
def answer_question(question: str, context: str = "") -> str:
    question = question.strip()
    context = context.strip()
    if context:
        result = qa_pipeline(question=question, context=context)
        return f'{result["answer"]}  (confidence: {result["score"]:.2f})'
    else:
        prompt = f"Answer the following question clearly and concisely: {question}"
        result = text2text(prompt, max_new_tokens=150, do_sample=False)
        return result[0]["generated_text"].strip()


# ---------- 4. Quiz Generation Module (/quiz) ----------
def generate_quiz(source_text: str, num_questions: int = 3) -> str:
    prompt = (
        f"Create {num_questions} multiple choice questions to test understanding of the "
        f"following content. For each question, give 4 options labeled A) B) C) D) and then "
        f"state 'Answer: <letter>' on its own line.\n\nContent: {source_text}"
    )
    result = text2text(prompt, max_new_tokens=400, do_sample=False)
    quiz_text = result[0]["generated_text"].strip()
    if "A)" not in quiz_text and "A." not in quiz_text:
        quiz_text = "(Loosely-formatted output; try flan-t5-large for stricter formatting.)\n\n" + quiz_text
    return quiz_text


# ---------- 5. Summarization Module (/summarize) ----------
def summarize_text(text: str, max_len: int = 130, min_len: int = 30) -> str:
    text = text.strip()
    if len(text.split()) < 20:
        return "Please provide a longer passage (at least ~20 words) to summarize."
    result = summarizer(text, max_length=max_len, min_length=min_len, do_sample=False)
    return result[0]["summary_text"].strip()


# ---------- 6. Learning Path Module (/learn/recommendations) ----------
def generate_learning_path(subject: str, level: str = "beginner", num_steps: int = 6) -> str:
    prompt = (
        f"Create a numbered, step-by-step learning path with {num_steps} sequential steps "
        f"for a {level} learner who wants to learn '{subject}'. For each step give a short "
        f"topic name and one sentence describing what to focus on."
    )
    result = text2text(prompt, max_new_tokens=350, do_sample=False)
    return result[0]["generated_text"].strip()


# ---------- 7. "FastAPI Backend" router ----------
def route_request(module: str, **kwargs) -> str:
    module = module.lower().strip()
    try:
        if module == "explain":
            return explain_topic(kwargs.get("topic", ""), kwargs.get("level", "beginner"))
        elif module == "qa":
            return answer_question(kwargs.get("question", ""), kwargs.get("context", ""))
        elif module == "quiz":
            return generate_quiz(kwargs.get("topic", ""), int(kwargs.get("num_questions", 3)))
        elif module == "summarize":
            return summarize_text(kwargs.get("text", ""))
        elif module == "learn":
            return generate_learning_path(kwargs.get("topic", ""), kwargs.get("level", "beginner"))
        else:
            return f"Unknown module: {module}"
    except Exception as e:
        return f"Error while processing request: {e}"


# ---------- 8. "Frontend" — interactive widget UI ----------
module_dropdown = widgets.Dropdown(
    options=[
        ("Explanation", "explain"), ("Q&A", "qa"), ("Quiz Generation", "quiz"),
        ("Summarization", "summarize"), ("Learning Path", "learn"),
    ],
    description="Module:", style={"description_width": "initial"},
)
level_dropdown = widgets.Dropdown(
    options=["beginner", "intermediate", "advanced"],
    description="Level:", style={"description_width": "initial"},
)
topic_box = widgets.Text(
    description="Topic:", placeholder="e.g. photosynthesis, machine learning",
    style={"description_width": "initial"}, layout=widgets.Layout(width="500px"),
)
question_box = widgets.Text(
    description="Question:", placeholder="(used for Q&A module)",
    style={"description_width": "initial"}, layout=widgets.Layout(width="500px"),
)
context_box = widgets.Textarea(
    description="Context / Text:", placeholder="(optional context for Q&A, or the passage to summarize)",
    style={"description_width": "initial"}, layout=widgets.Layout(width="500px", height="100px"),
)
num_questions_box = widgets.BoundedIntText(
    value=3, min=1, max=10, description="# Quiz Qs:", style={"description_width": "initial"},
)
run_button = widgets.Button(description="Run EduGenie \u25b6", button_style="success")
output_area = widgets.Output()

def on_run_clicked(b):
    with output_area:
        clear_output()
        module = module_dropdown.value
        display(Markdown(f"### EduGenie -- {module_dropdown.label} results"))
        try:
            if module == "explain":
                result = route_request("explain", topic=topic_box.value, level=level_dropdown.value)
            elif module == "qa":
                result = route_request("qa", question=question_box.value, context=context_box.value)
            elif module == "quiz":
                source = context_box.value.strip() or topic_box.value
                result = route_request("quiz", topic=source, num_questions=num_questions_box.value)
            elif module == "summarize":
                result = route_request("summarize", text=context_box.value)
            elif module == "learn":
                result = route_request("learn", topic=topic_box.value, level=level_dropdown.value)
            else:
                result = "Unknown module."
        except Exception as e:
            result = f"Something went wrong: {type(e).__name__}: {e}"
        display(Markdown(result))
        display(Markdown("---\n*Pick another module above and click Run again to continue.*"))

run_button.on_click(on_run_clicked)

ui = widgets.VBox([
    widgets.HTML("<h3>EduGenie -- Frontend: User Input</h3>"),
    module_dropdown, level_dropdown, topic_box, question_box, context_box,
    num_questions_box, run_button, output_area,
])
display(ui)


## Notes

- **No API keys anywhere** — every model is public on the Hugging Face Hub, loaded locally via `transformers.pipeline(...)`.
- **First run is slower** because ~2 GB of models download once. On later runs in the *same* Colab session, the models are already loaded in memory (don't re-run Cell 2 unless the runtime disconnected).
- **Even better quality**: swap `google/flan-t5-base` → `google/flan-t5-large` in Cell 2 if you have a strong GPU and don't mind an even longer first download.
- This notebook never installs/upgrades `torch` or `torchvision`, so it runs on a fresh Colab runtime without needing any restart.
